# Gom dữ liệu trên Hugging Face: keyframe mới + cũ vào MỘT thư mục, video cũ vào CHUNG thư mục video mới

Chạy trên Colab (không tốn mạng máy bạn). Kết quả trên `5clnl/idk`:

| Thư mục | Nội dung |
|---|---|
| `keyframes_all/` | **khoảng 8 file zip** chứa toàn bộ keyframe: bộ mới M, N+S và bộ cũ L21–L30 (kể cả 11 video bổ sung), kèm `manifest.json` và `README.md` |
| `videos/` | 614 video mới **+ 873 video cũ** (lấy ra từ `videos.zip`) — một thư mục |
| `index_all/` | chỉ mục PE-Core của tất cả video (được upload từ máy chủ nhà, không do notebook này làm) |

Ảnh trong các zip nằm phẳng `<video>_frame_<số frame>.jpg`: tải hết `keyframes_all/`, giải nén tất cả vào **cùng một thư mục** là xong.

**Trước khi chạy:** `Runtime → Change runtime type` → CPU là đủ (không cần GPU). Cần khoảng **30 GB ổ trống** (keyframe cũ chỉ cần nếu không chép được trên máy chủ HF).

| Cell | Việc | Thời gian ước tính |
|---|---|---|
| 1 | Đăng nhập Hugging Face (token quyền **Write**) | |
| 2 | Cấu hình (thường không cần sửa) | |
| 3 | Gộp keyframe → `keyframes_all/` | 20–40 phút |
| 4 | Video cũ → `videos/` (83 GB) | khoảng 30–60 phút |

Cell 3 và 4 độc lập; bị ngắt thì **chạy lại đúng cell đó**: phần đã lên HF được bỏ qua. Các file cũ (`keyframes_batch2/`, `keyframes.zip`, `videos.zip`…) **không bị xóa**.


In [ ]:
#@title 1. Đăng nhập Hugging Face
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 2. Cấu hình
REPO = "5clnl/idk"
# --- keyframe
OUT_DIR = "keyframes_all"                  # thư mục đích duy nhất cho keyframe
PER_VIDEO_DIRS = ["keyframes_batch2", "keyframes_S01", "keyframes_old_missing"]   # các zip từng video (bộ mới + 11 video cũ bổ sung)
OLD_ZIP = "keyframes.zip"                  # keyframe bộ cũ L21-L30 (gốc repo)
PART_GB = 9.5                              # mỗi file zip tối đa ~9,5 GB
COPY_OLD_SERVER_SIDE = True                # chép keyframes.zip ngay trên máy chủ HF (vài giây); không được thì tự tải về chia phần
DOWNLOAD_WORKERS = 8
# --- video
VIDEO_ZIP = "videos.zip"                   # 873 video cũ
VIDEO_DIR = "videos"                       # thư mục video chung (đang có 614 video mới)
EXTRACT_WORKERS = 6                        # số video giải nén song song (mỗi luồng đọc videos.zip bằng HTTP Range)
UPLOAD_BATCH = 25                          # upload mỗi lô ... video
UPLOAD_BATCH_GB = 4
STAGE_MAX = 40                             # tối đa ... video chờ upload trên ổ Colab
print(f"{REPO}: keyframe -> {OUT_DIR}/ (mỗi file <= {PART_GB} GB), video cũ -> {VIDEO_DIR}/")


In [ ]:
#@title 3. Gộp TẤT CẢ keyframe (mới + cũ) vào MỘT thư mục: keyframes_all/ (khoảng 8 file zip)
import itertools, json, os, re, shutil, time, zipfile
from collections import OrderedDict, deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import CommitOperationAdd, CommitOperationCopy, HfApi, HfFileSystem, hf_hub_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi()
WORK = Path("/content/gop_kf")
WORK.mkdir(parents=True, exist_ok=True)
CAP = int(PART_GB * 1e9)
FRAME_RE = re.compile(r"^(.+?)_frame_(\d+)\.jpg$")


def log(msg):
    print(msg, flush=True)


def sizes_of(paths):
    out = {}
    paths = list(paths)
    for i in range(0, len(paths), 200):
        for x in api.get_paths_info(REPO, paths[i:i + 200], repo_type="dataset"):
            out[x.path] = x.size
    return out


files = api.list_repo_files(REPO, repo_type="dataset")
fileset = set(files)

# ---------------------------------------------------------------- nguồn: mỗi video một zip (mới + 11 video cũ bổ sung)
per_video = {}
for f in files:
    if f.endswith(".zip") and f.startswith(tuple(d + "/" for d in PER_VIDEO_DIRS)):
        v = Path(f).stem
        if v in per_video:
            raise RuntimeError(f"video {v} xuất hiện ở cả {per_video[v]} và {f}")
        per_video[v] = f
src_size = sizes_of(sorted(per_video.values()))

expected = {}                                   # số ảnh mong đợi của từng video (phát hiện zip thiếu)
if "keyframes_batch2/index.json" in fileset:
    p = hf_hub_download(REPO, "keyframes_batch2/index.json", repo_type="dataset", local_dir=str(WORK / "meta"))
    expected.update({v: x["frames"] for v, x in json.load(open(p))["videos"].items()})
for f in files:
    if f.startswith("keyframes_S01/") and f.endswith(".json"):
        p = hf_hub_download(REPO, f, repo_type="dataset", local_dir=str(WORK / "meta"))
        expected[Path(f).stem] = json.load(open(p))["keyframes"]


def group_of(v):
    d = per_video[v].split("/")[0]
    if d == "keyframes_old_missing":
        return "old_extra"
    return "M" if v.startswith("M") else "N_S"


def pack(items, size):
    """Chia danh sách (giữ thứ tự) thành các phần, mỗi phần tổng dung lượng <= PART_GB."""
    parts, cur, cur_b = [], [], 0
    for it in items:
        b = size(it)
        if cur and cur_b + b > CAP:
            parts.append(cur)
            cur, cur_b = [], 0
        cur.append(it)
        cur_b += b
    if cur:
        parts.append(cur)
    return parts


plan = []                                       # (tên file, danh sách video)
for g in ("M", "N_S", "old_extra"):
    vids = sorted(v for v in per_video if group_of(v) == g)
    ps = pack(vids, lambda v: src_size[per_video[v]])
    plan += [(f"keyframes_{g}_{i}of{len(ps)}.zip", p) for i, p in enumerate(ps, 1)]
log(f"{len(per_video)} zip từng video -> {len(plan)} file: " + ", ".join(n for n, _ in plan))


def remote_has(name):
    return f"{OUT_DIR}/{name}" in fileset


def upload_verified(local, name, msg):
    size = Path(local).stat().st_size
    for attempt in range(1, 6):
        try:
            log(f"   upload {name} ({size / 1e9:.2f} GB), lần {attempt} ...")
            api.upload_file(path_or_fileobj=str(local), path_in_repo=f"{OUT_DIR}/{name}", repo_id=REPO, repo_type="dataset",
                            commit_message=msg)
            got = sizes_of([f"{OUT_DIR}/{name}"]).get(f"{OUT_DIR}/{name}")
            if got == size:
                fileset.add(f"{OUT_DIR}/{name}")
                return
            log(f"   kích thước trên HF {got} khác {size} -> thử lại")
        except Exception as e:
            log(f"   upload lỗi: {type(e).__name__}: {str(e)[:200]}")
        time.sleep(20 * attempt)
    raise RuntimeError(f"upload {name} không thành công")


def verify_zip(path, counts):
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        bad = z.testzip()                      # đọc lại toàn bộ, kiểm CRC từng ảnh
    if bad is not None:
        raise RuntimeError(f"{path.name}: ảnh {bad} hỏng")
    if len(names) != len(set(names)) or len(names) != sum(counts.values()):
        raise RuntimeError(f"{path.name}: {len(names)} mục, đáng lẽ {sum(counts.values())} ảnh không trùng")


def listing_counts(repo_path):
    """Số ảnh từng video trong một zip trên HF, chỉ đọc mục lục (vài giây, không tải cả file)."""
    for attempt in range(1, 4):
        try:
            with HfFileSystem().open(f"datasets/{REPO}/{repo_path}", "rb", block_size=4 << 20) as fh:
                names = zipfile.ZipFile(fh).namelist()
            break
        except Exception as e:
            if attempt == 3:
                raise
            log(f"   đọc mục lục {repo_path} lỗi ({type(e).__name__}) -> thử lại")
            time.sleep(10 * attempt)
    counts = OrderedDict()
    for n in sorted(names):
        m = FRAME_RE.match(n.split("/")[-1])
        if m:
            counts[m.group(1)] = counts.get(m.group(1), 0) + 1
    return counts, len(names)


def fetch(v):
    for attempt in range(1, 6):
        try:
            return hf_hub_download(REPO, per_video[v], repo_type="dataset", local_dir=str(WORK / "dl"))
        except Exception as e:
            log(f"[{v}] tải lỗi lần {attempt}/5: {type(e).__name__}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"không tải được {per_video[v]}")


def build_from_videos(name, vids):
    out_path, tmp = WORK / name, WORK / (name + ".tmp")
    counts, t0 = OrderedDict(), time.time()
    with ThreadPoolExecutor(DOWNLOAD_WORKERS) as ex, zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED, allowZip64=True) as out:
        it = iter(vids)
        pending = deque((v, ex.submit(fetch, v)) for v in itertools.islice(it, DOWNLOAD_WORKERS * 2))
        while pending:
            v, fut = pending.popleft()
            p = fut.result()
            nxt = next(it, None)
            if nxt is not None:
                pending.append((nxt, ex.submit(fetch, nxt)))
            with zipfile.ZipFile(p) as zf:
                names = sorted(n for n in zf.namelist() if n.lower().endswith(".jpg"))
                wrong = [n for n in names if not (FRAME_RE.match(n) and FRAME_RE.match(n).group(1) == v)]
                if wrong or not names:
                    raise RuntimeError(f"{v}: tên ảnh không đúng dạng {v}_frame_NNNNNN.jpg ({wrong[:2]})")
                if v in expected and len(names) != expected[v]:
                    raise RuntimeError(f"{v}: zip có {len(names)} ảnh, đáng lẽ {expected[v]}")
                for n in names:
                    zi = zipfile.ZipInfo(n, date_time=zf.getinfo(n).date_time)
                    zi.compress_type = zipfile.ZIP_STORED
                    out.writestr(zi, zf.read(n))
                counts[v] = len(names)
            os.remove(p)
            if len(counts) % 50 == 0:
                log(f"   {name}: {len(counts)}/{len(vids)} video, {(time.time() - t0) / 60:.1f} phút")
    os.replace(tmp, out_path)
    verify_zip(out_path, counts)
    upload_verified(out_path, name, f"keyframes_all: {name} ({sum(counts.values())} ảnh)")
    os.remove(out_path)
    return counts


manifest, t_all = OrderedDict(), time.time()

# ---------------------------------------------------------------- 1) keyframe mới (M, N+S) và 11 video cũ bổ sung
for name, vids in plan:
    if remote_has(name):
        log(f"[bỏ qua] {name}: đã có trên HF")
        manifest[name] = listing_counts(f"{OUT_DIR}/{name}")[0]
        continue
    log(f"[{name}] gộp {len(vids)} video ...")
    manifest[name] = build_from_videos(name, vids)
    log(f"[{name}] xong: {sum(manifest[name].values()):,} ảnh ({(time.time() - t_all) / 60:.1f} phút từ đầu)")

# ---------------------------------------------------------------- 2) keyframe cũ L21-L30 (keyframes.zip ở gốc repo)
OLD_NAME = "keyframes_old_L21-L30.zip"
old_parts = sorted(f.split("/")[-1] for f in fileset if f.startswith(f"{OUT_DIR}/keyframes_old_") and re.search(r"_\d+of\d+\.zip$", f)
                   and not f.split("/")[-1].startswith("keyframes_old_extra"))
if OLD_ZIP not in fileset and not remote_has(OLD_NAME) and not old_parts:
    log(f"!! Không thấy {OLD_ZIP} trên HF — bỏ qua phần keyframe cũ")
elif remote_has(OLD_NAME):
    log(f"[bỏ qua] {OLD_NAME}: đã có trên HF")
    manifest[OLD_NAME] = listing_counts(f"{OUT_DIR}/{OLD_NAME}")[0]
else:
    copied = False
    if not old_parts and COPY_OLD_SERVER_SIDE:
        # chép ngay trên máy chủ HF: không tải xuống, không tải lên, xong trong vài giây
        try:
            api.create_commit(repo_id=REPO, repo_type="dataset", commit_message=f"keyframes_all: {OLD_NAME} (chép từ {OLD_ZIP})",
                              operations=[CommitOperationCopy(src_path_in_repo=OLD_ZIP, path_in_repo=f"{OUT_DIR}/{OLD_NAME}")])
            copied = sizes_of([OLD_ZIP, f"{OUT_DIR}/{OLD_NAME}"])
            copied = copied.get(OLD_ZIP) == copied.get(f"{OUT_DIR}/{OLD_NAME}") and copied.get(OLD_ZIP) is not None
            if copied:
                fileset.add(f"{OUT_DIR}/{OLD_NAME}")
                manifest[OLD_NAME] = listing_counts(f"{OUT_DIR}/{OLD_NAME}")[0]
                log(f"[{OLD_NAME}] đã chép trên máy chủ HF: {sum(manifest[OLD_NAME].values()):,} ảnh")
        except Exception as e:
            log(f"   chép trên máy chủ không được ({type(e).__name__}: {str(e)[:150]}) -> tải về rồi chia phần")
    if not copied:
        local = Path(hf_hub_download(REPO, OLD_ZIP, repo_type="dataset", local_dir=str(WORK / "dl")))
        with zipfile.ZipFile(local) as zf:
            by_video = OrderedDict()
            for zi in sorted(zf.infolist(), key=lambda z: z.filename):
                m = FRAME_RE.match(zi.filename.split("/")[-1])
                if m:
                    by_video.setdefault(m.group(1), []).append(zi)
            ps = pack(list(by_video), lambda v: sum(z.file_size for z in by_video[v]))
            for i, vids in enumerate(ps, 1):
                name = f"keyframes_old_{i}of{len(ps)}.zip"
                if remote_has(name):
                    log(f"[bỏ qua] {name}: đã có trên HF")
                    manifest[name] = listing_counts(f"{OUT_DIR}/{name}")[0]
                    continue
                out_path, tmp, counts = WORK / name, WORK / (name + ".tmp"), OrderedDict()
                with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED, allowZip64=True) as out:
                    for v in vids:
                        for zi in by_video[v]:
                            ni = zipfile.ZipInfo(zi.filename.split("/")[-1], date_time=zi.date_time)
                            ni.compress_type = zipfile.ZIP_STORED
                            out.writestr(ni, zf.read(zi))
                        counts[v] = len(by_video[v])
                os.replace(tmp, out_path)
                verify_zip(out_path, counts)
                upload_verified(out_path, name, f"keyframes_all: {name} ({sum(counts.values())} ảnh)")
                os.remove(out_path)
                manifest[name] = counts
                log(f"[{name}] xong: {sum(counts.values()):,} ảnh")
        os.remove(local)

# ---------------------------------------------------------------- 3) manifest + hướng dẫn, kiểm tra không trùng video
seen = {}
for part, vids in manifest.items():
    for v in vids:
        if v in seen:
            raise RuntimeError(f"video {v} nằm ở cả {seen[v]} và {part}")
        seen[v] = part
total = sum(n or 0 for vids in manifest.values() for n in vids.values())
listing = "\n".join(f"| `{p}` | {len(v)} | {sum(n or 0 for n in v.values()):,} |" for p, v in manifest.items())
readme = f"""# keyframes_all — toàn bộ keyframe (bộ mới M/N/S + bộ cũ L21–L30) trong MỘT thư mục

{len(manifest)} file zip, {len(seen):,} video, {total:,} ảnh. Bên trong mỗi zip ảnh nằm phẳng, tên `<video>_frame_<số frame 6 chữ số>.jpg`;
giải nén TẤT CẢ vào cùng một thư mục là dùng được ngay (không có thư mục con, không trùng tên giữa các zip).

| File | Video | Ảnh |
|---|---|---|
{listing}

Tải và giải nén (Linux):

```bash
pip install -U huggingface_hub
hf download {REPO} --repo-type dataset --include "{OUT_DIR}/*" --local-dir data
mkdir -p data/keyframes && for f in data/{OUT_DIR}/*.zip; do unzip -q -o "$f" -d data/keyframes; done
```

Cùng repo: `videos/` = toàn bộ video (mới + cũ, một thư mục), `index_all/` = chỉ mục PE-Core của tất cả video (emb.npy, frames.npy, vid.npy, ms.npy, videos.json).
`manifest.json` liệt kê từng video nằm trong file zip nào và có bao nhiêu ảnh.
"""
ops = [CommitOperationAdd(path_in_repo=f"{OUT_DIR}/manifest.json",
                          path_or_fileobj=json.dumps({"parts": manifest, "videos": len(seen), "images": total}, ensure_ascii=False, indent=1).encode("utf-8")),
       CommitOperationAdd(path_in_repo=f"{OUT_DIR}/README.md", path_or_fileobj=readme.encode("utf-8"))]
api.create_commit(repo_id=REPO, repo_type="dataset", operations=ops, commit_message="keyframes_all: manifest + README")
log("\n================ XONG PHẦN KEYFRAME ================")
log(f"{OUT_DIR}/ có {len(manifest)} file zip: {len(seen):,} video, {total:,} ảnh ({(time.time() - t_all) / 60:.1f} phút)")
for p, v in manifest.items():
    log(f"  {p}: {len(v)} video, {sum(n or 0 for n in v.values()):,} ảnh")
log(f"Tải cả thư mục: hf download {REPO} --repo-type dataset --include \"{OUT_DIR}/*\" --local-dir data")


In [ ]:
#@title 4. Đưa 873 video cũ (trong videos.zip) vào CHUNG thư mục videos/ với 614 video mới
import queue, shutil, threading, time, zipfile
from pathlib import Path
from huggingface_hub import HfApi, HfFileSystem
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi()
VW = Path("/content/gop_video")
VW.mkdir(parents=True, exist_ok=True)
VIDEO_EXT = (".mp4", ".mov", ".mkv", ".avi", ".webm")


def log(msg):
    print(msg, flush=True)


def open_zip():
    """videos.zip trên HF, đọc từng đoạn (HTTP Range) — không tải cả 83 GB về Colab."""
    return zipfile.ZipFile(HfFileSystem().open(f"datasets/{REPO}/{VIDEO_ZIP}", "rb", block_size=16 << 20))


all_files = api.list_repo_files(REPO, repo_type="dataset")
remote = {}
vpaths = [f for f in all_files if f.startswith(VIDEO_DIR + "/")]
for i in range(0, len(vpaths), 200):
    for x in api.get_paths_info(REPO, vpaths[i:i + 200], repo_type="dataset"):
        remote[x.path] = x.size

entries = [zi for zi in open_zip().infolist() if not zi.is_dir() and zi.filename.lower().endswith(VIDEO_EXT)]
todo, clash = [], []
for zi in entries:
    dst = f"{VIDEO_DIR}/{Path(zi.filename).name}"
    if dst not in remote:
        todo.append(zi)
    elif remote[dst] != zi.file_size:
        clash.append(dst)                    # một video KHÁC trùng tên: không ghi đè
log(f"videos.zip: {len(entries)} video cũ | {VIDEO_DIR}/ trên HF đang có {len(vpaths)} file | cần đưa lên {len(todo)} "
    f"({sum(z.file_size for z in todo) / 1e9:.1f} GB)" + (f" | TRÙNG TÊN khác nội dung (bỏ qua): {clash[:5]}" if clash else ""))

work_q, done_q = queue.Queue(), queue.Queue()
for zi in todo:
    work_q.put(zi)
slots = threading.BoundedSemaphore(max(STAGE_MAX, UPLOAD_BATCH + EXTRACT_WORKERS))   # số video đã giải nén chờ upload (ổ đĩa Colab)


def extractor():
    zf = None
    while True:
        try:
            zi = work_q.get_nowait()
        except queue.Empty:
            return
        slots.acquire()
        name = Path(zi.filename).name
        out, part = VW / name, VW / (name + ".part")
        err = None
        for attempt in range(1, 5):
            try:
                if zf is None:
                    zf = open_zip()
                with zf.open(zi) as src, open(part, "wb") as dst:       # zipfile tự kiểm CRC32 khi đọc hết mục
                    shutil.copyfileobj(src, dst, 4 << 20)
                if part.stat().st_size != zi.file_size:
                    raise IOError(f"kích thước {part.stat().st_size} khác {zi.file_size}")
                part.replace(out)
                err = None
                break
            except Exception as e:
                err = e
                zf = None
                part.unlink(missing_ok=True)
                time.sleep(5 * attempt)
        if err is None:
            done_q.put((name, zi.file_size, None))
        else:
            slots.release()
            done_q.put((name, zi.file_size, f"{type(err).__name__}: {str(err)[:150]}"))


def upload_batch(batch):
    names = [n for n, _ in batch]
    for attempt in range(1, 6):
        try:
            api.upload_folder(repo_id=REPO, repo_type="dataset", folder_path=str(VW), path_in_repo=VIDEO_DIR,
                              allow_patterns=names, commit_message=f"videos: +{len(names)} video cũ từ {VIDEO_ZIP}")
            got = {}
            for x in api.get_paths_info(REPO, [f"{VIDEO_DIR}/{n}" for n in names], repo_type="dataset"):
                got[x.path] = x.size
            bad = [n for n, s in batch if got.get(f"{VIDEO_DIR}/{n}") != s]
            if not bad:
                return True
            log(f"   {len(bad)} file sai kích thước sau upload (lần {attempt}): {bad[:3]}")
        except Exception as e:
            log(f"   upload lỗi lần {attempt}: {type(e).__name__}: {str(e)[:200]}")
        time.sleep(20 * attempt)
    return False


threads = [threading.Thread(target=extractor, daemon=True) for _ in range(max(1, EXTRACT_WORKERS))]
for t in threads:
    t.start()
batch, failed, t0 = [], [], time.time()
stat = {"n": 0, "gb": 0.0}


def flush():
    global batch
    if not batch:
        return
    ok = upload_batch(batch)
    for n, s in batch:
        (VW / n).unlink(missing_ok=True)
        slots.release()
    if ok:
        stat["n"] += len(batch)
        stat["gb"] += sum(s for _, s in batch) / 1e9
    else:
        failed.extend(n for n, _ in batch)
    batch = []
    el = time.time() - t0
    log(f"  đã đưa lên {stat['n']}/{len(todo)} video ({stat['gb']:.1f} GB) | {el / 60:.1f} phút | "
        f"còn ≈ {el / max(stat['n'], 1) * (len(todo) - stat['n'] - len(failed)) / 60:.0f} phút | lỗi {len(failed)}")


remaining, idle = len(todo), 0
while remaining:
    try:
        name, size, err = done_q.get(timeout=60)
        idle = 0
    except queue.Empty:
        flush()                                      # không chờ đủ lô: đưa lên những gì đã có
        idle += 1
        if idle % 10 == 0:
            log(f"!! {idle} phút không giải nén xong video nào — mạng hoặc HF đang chậm; vẫn đang chờ")
        continue
    remaining -= 1
    if err:
        failed.append(name)
        log(f"[{name}] !! LỖI: {err}")
    else:
        batch.append((name, size))
    if len(batch) >= UPLOAD_BATCH or sum(s for _, s in batch) >= UPLOAD_BATCH_GB * 1e9 or remaining == 0:
        flush()
for t in threads:
    t.join(timeout=5)

log("\n================ XONG PHẦN VIDEO ================")
log(f"Đưa lên {stat['n']} video cũ vào {VIDEO_DIR}/ ({stat['gb']:.1f} GB, {(time.time() - t0) / 60:.1f} phút)")
if failed:
    log("  Lỗi (chạy lại cell này để làm nốt): " + ", ".join(sorted(failed)[:40]))
else:
    log(f"{VIDEO_DIR}/ giờ có đủ video mới + cũ. Tải cả thư mục: hf download {REPO} --repo-type dataset --include \"{VIDEO_DIR}/*\" --local-dir data")
